In [1]:
import configparser
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    ArrayType,
    StructType,
    StructField,
    IntegerType,
    StringType
)

In [2]:
config = configparser.RawConfigParser()
config_path = Path.cwd() / "all_conn"

if not config.read(config_path):
    raise FileNotFoundError(f"Configuration file not found: {config_path}")

endpoint_url = config.get("conn", "endpoint_url")
aws_access_key_id = config.get("conn", "aws_access_key_id")
aws_secret_access_key = config.get("conn", "aws_secret_access_key")

In [3]:
spark = (
    SparkSession.builder
    .appName("Transform Data")
    .config("spark.jars.packages","org.apache.hadoop:hadoop-aws:3.3.4,""io.delta:delta-spark_2.12:3.3.2")
    .config("spark.sql.extensions","io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog","org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.hadoop.fs.s3a.endpoint", endpoint_url)
    .config("spark.hadoop.fs.s3a.access.key", aws_access_key_id)
    .config("spark.hadoop.fs.s3a.secret.key", aws_secret_access_key)
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .getOrCreate()
)

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-08b0eaa6-7c6e-4070-b2e2-5a519d77d58d;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
	found io.delta#delta-spark_2.12;3.3.2 in central
	found io.delta#delta-storage;3.3.2 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
downloading https://repo1.maven.org/maven2/io/delta/delta-spark_2.12/3.3.2/delta-spark_2.12-3.3.2.jar ...
	[SUCCESSFUL ] io.delta#delta-spark_2.12;3.3.2!delta-spark_2.12.jar (3551ms)
downloading https://repo1.maven.org/maven2/io/delta/delta-storage/3.3.2/delta-storage-3.3.2.jar ...
	[SUCCESSFUL ] io.delta#delta-storage;3.3.2!delta-storage.jar (204ms)

In [4]:
credits_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("quote", '"')
    .option("escape", '"')
    .option("multiLine", "true")
    .csv("s3a://tmdb-bronze/tmdb_5000_credits.csv")
)

26/07/05 15:54:19 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


In [5]:
credits_df.printSchema()
credits_df.show(5)
print("Row Numbers:", credits_df.count())

root
 |-- movie_id: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- cast: string (nullable = true)
 |-- crew: string (nullable = true)

+--------+--------------------+--------------------+--------------------+
|movie_id|               title|                cast|                crew|
+--------+--------------------+--------------------+--------------------+
|   19995|              Avatar|[{"cast_id": 242,...|[{"credit_id": "5...|
|     285|Pirates of the Ca...|[{"cast_id": 4, "...|[{"credit_id": "5...|
|  206647|             Spectre|[{"cast_id": 1, "...|[{"credit_id": "5...|
|   49026|The Dark Knight R...|[{"cast_id": 2, "...|[{"credit_id": "5...|
|   49529|         John Carter|[{"cast_id": 5, "...|[{"credit_id": "5...|
+--------+--------------------+--------------------+--------------------+
only showing top 5 rows



Row Numbers: 4803


In [6]:
cast_schema = ArrayType(
    StructType([
        StructField("cast_id", IntegerType(), True),
        StructField("character", StringType(), True),
        StructField("credit_id", StringType(), True),
        StructField("gender", IntegerType(), True),
        StructField("id", IntegerType(), True),
        StructField("name", StringType(), True),
        StructField("order", IntegerType(), True)
    ])
)

In [7]:
cast_df = (
    credits_df
    .withColumn("cast_parsed", F.from_json(F.col("cast"), cast_schema)
    )
    .select("movie_id", "title", F.inline_outer(F.col("cast_parsed"))
    )
)

In [8]:
cast_df.show(10, truncate=False)

+--------+------+-------+-------------------+------------------------+------+-----+------------------+-----+
|movie_id|title |cast_id|character          |credit_id               |gender|id   |name              |order|
+--------+------+-------+-------------------+------------------------+------+-----+------------------+-----+
|19995   |Avatar|242    |Jake Sully         |5602a8a7c3a3685532001c9a|2     |65731|Sam Worthington   |0    |
|19995   |Avatar|3      |Neytiri            |52fe48009251416c750ac9cb|1     |8691 |Zoe Saldana       |1    |
|19995   |Avatar|25     |Dr. Grace Augustine|52fe48009251416c750aca39|1     |10205|Sigourney Weaver  |2    |
|19995   |Avatar|4      |Col. Quaritch      |52fe48009251416c750ac9cf|2     |32747|Stephen Lang      |3    |
|19995   |Avatar|5      |Trudy Chacon       |52fe48009251416c750ac9d3|1     |17647|Michelle Rodriguez|4    |
|19995   |Avatar|8      |Selfridge          |52fe48009251416c750ac9e1|2     |1771 |Giovanni Ribisi   |5    |
|19995   |Avatar|7 

In [9]:
cast_df.printSchema()

root
 |-- movie_id: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- cast_id: integer (nullable = true)
 |-- character: string (nullable = true)
 |-- credit_id: string (nullable = true)
 |-- gender: integer (nullable = true)
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- order: integer (nullable = true)



In [10]:
cast_df.count()

106300

## Missing Values --> Cast Table

In [11]:
cast_df.select([
    F.count(F.when(F.col(field).isNull(), field)).alias(field)
    for field in cast_df.columns
]).show()

+--------+-----+-------+---------+---------+------+---+----+-----+
|movie_id|title|cast_id|character|credit_id|gender| id|name|order|
+--------+-----+-------+---------+---------+------+---+----+-----+
|       0|    0|     43|       43|       43|    43| 43|  43|   43|
+--------+-----+-------+---------+---------+------+---+----+-----+



In [12]:
cast_df.filter(
    F.col("cast_id").isNull() &
    F.col("character").isNull() &
    F.col("credit_id").isNull() &
    F.col("gender").isNull() &
    F.col("id").isNull() &
    F.col("name").isNull() &
    F.col("order").isNull()
).show(50, truncate=False)
#count()

+--------+-----------------------------------------------------+-------+---------+---------+------+----+----+-----+
|movie_id|title                                                |cast_id|character|credit_id|gender|id  |name|order|
+--------+-----------------------------------------------------+-------+---------+---------+------+----+----+-----+
|17644   |Barney's Great Adventure                             |NULL   |NULL     |NULL     |NULL  |NULL|NULL|NULL |
|447027  |Running Forever                                      |NULL   |NULL     |NULL     |NULL  |NULL|NULL|NULL |
|346081  |Sardaarji                                            |NULL   |NULL     |NULL     |NULL  |NULL|NULL|NULL |
|126509  |2016: Obama's America                                |NULL   |NULL     |NULL     |NULL  |NULL|NULL|NULL |
|371085  |Sharkskin                                            |NULL   |NULL     |NULL     |NULL  |NULL|NULL|NULL |
|325140  |Hum To Mohabbat Karega                               |NULL   |

In [13]:
duplicate_count = (
    cast_df.count()
    - cast_df.dropDuplicates().count()
)

print("Duplicate row numbers:", duplicate_count)

Duplicate row numbers: 0


In [14]:
cast_df = cast_df.fillna(
    {"credit_id": "0000000000"}
)

In [15]:
cast_df.filter(
    F.col("cast_id").isNull() &
    F.col("character").isNull() &
    F.col("gender").isNull() &
    F.col("id").isNull() &
    F.col("name").isNull() &
    F.col("order").isNull()
).show(50, truncate=False)

+--------+-----------------------------------------------------+-------+---------+----------+------+----+----+-----+
|movie_id|title                                                |cast_id|character|credit_id |gender|id  |name|order|
+--------+-----------------------------------------------------+-------+---------+----------+------+----+----+-----+
|17644   |Barney's Great Adventure                             |NULL   |NULL     |0000000000|NULL  |NULL|NULL|NULL |
|447027  |Running Forever                                      |NULL   |NULL     |0000000000|NULL  |NULL|NULL|NULL |
|346081  |Sardaarji                                            |NULL   |NULL     |0000000000|NULL  |NULL|NULL|NULL |
|126509  |2016: Obama's America                                |NULL   |NULL     |0000000000|NULL  |NULL|NULL|NULL |
|371085  |Sharkskin                                            |NULL   |NULL     |0000000000|NULL  |NULL|NULL|NULL |
|325140  |Hum To Mohabbat Karega                               |

## Crew Table

In [16]:
crew_schema = ArrayType(
    StructType([
        StructField("credit_id", StringType(), True),
        StructField("department", StringType(), True),
        StructField("gender", IntegerType(), True),
        StructField("id", IntegerType(), True),
        StructField("job", StringType(), True),
        StructField("name", StringType(), True)
    ])
)

In [17]:
crew_df = (
    credits_df
    .withColumn("crew_parsed", F.from_json(F.col("crew"), crew_schema)
    )
    .select("movie_id","title", F.inline_outer("crew_parsed")
    )
)

In [18]:
crew_df.show(10, truncate=False)

+--------+------+------------------------+----------+------+----+------------------------+-----------------+
|movie_id|title |credit_id               |department|gender|id  |job                     |name             |
+--------+------+------------------------+----------+------+----+------------------------+-----------------+
|19995   |Avatar|52fe48009251416c750aca23|Editing   |0     |1721|Editor                  |Stephen E. Rivkin|
|19995   |Avatar|539c47ecc3a36810e3001f87|Art       |2     |496 |Production Design       |Rick Carter      |
|19995   |Avatar|54491c89c3a3680fb4001cf7|Sound     |0     |900 |Sound Designer          |Christopher Boyes|
|19995   |Avatar|54491cb70e0a267480001bd0|Sound     |0     |900 |Supervising Sound Editor|Christopher Boyes|
|19995   |Avatar|539c4a4cc3a36810c9002101|Production|1     |1262|Casting                 |Mali Finn        |
|19995   |Avatar|5544ee3b925141499f0008fc|Sound     |2     |1729|Original Music Composer |James Horner     |
|19995   |Avatar|52

In [19]:
crew_df.printSchema()

root
 |-- movie_id: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- credit_id: string (nullable = true)
 |-- department: string (nullable = true)
 |-- gender: integer (nullable = true)
 |-- id: integer (nullable = true)
 |-- job: string (nullable = true)
 |-- name: string (nullable = true)



In [20]:
crew_df.count()

129609

## Missing Value

In [21]:
crew_df.select([
    F.count(F.when(F.col(field).isNull(), field)).alias(field)
    for field in crew_df.columns
]).show()

+--------+-----+---------+----------+------+---+---+----+
|movie_id|title|credit_id|department|gender| id|job|name|
+--------+-----+---------+----------+------+---+---+----+
|       0|    0|       28|        28|    28| 28| 28|  28|
+--------+-----+---------+----------+------+---+---+----+



In [22]:
crew_df.filter(
    F.col("credit_id").isNull() &
    F.col("department").isNull() &
    F.col("gender").isNull() &
    F.col("id").isNull() &
    F.col("job").isNull() &
    F.col("name").isNull()
).show(30, truncate=False)
#count()

+--------+-----------------------------------------------------+---------+----------+------+----+----+----+
|movie_id|title                                                |credit_id|department|gender|id  |job |name|
+--------+-----------------------------------------------------+---------+----------+------+----+----+----+
|19615   |Flying By                                            |NULL     |NULL      |NULL  |NULL|NULL|NULL|
|447027  |Running Forever                                      |NULL     |NULL      |NULL  |NULL|NULL|NULL|
|55831   |Boynton Beach Club                                   |NULL     |NULL      |NULL  |NULL|NULL|NULL|
|371085  |Sharkskin                                            |NULL     |NULL      |NULL  |NULL|NULL|NULL|
|48382   |The Book of Mormon Movie, Volume 1: The Journey      |NULL     |NULL      |NULL  |NULL|NULL|NULL|
|325140  |Hum To Mohabbat Karega                               |NULL     |NULL      |NULL  |NULL|NULL|NULL|
|20653   |Roadside Romeo    

In [23]:
crew_df = crew_df.fillna(
    {
        "credit_id" : "0000000000"
    }
)

In [24]:
crew_df.filter(
    F.col("department").isNull() &
    F.col("gender").isNull() &
    F.col("id").isNull() &
    F.col("job").isNull() &
    F.col("name").isNull()
).show(30, truncate=False)
#count()

+--------+-----------------------------------------------------+----------+----------+------+----+----+----+
|movie_id|title                                                |credit_id |department|gender|id  |job |name|
+--------+-----------------------------------------------------+----------+----------+------+----+----+----+
|19615   |Flying By                                            |0000000000|NULL      |NULL  |NULL|NULL|NULL|
|447027  |Running Forever                                      |0000000000|NULL      |NULL  |NULL|NULL|NULL|
|55831   |Boynton Beach Club                                   |0000000000|NULL      |NULL  |NULL|NULL|NULL|
|371085  |Sharkskin                                            |0000000000|NULL      |NULL  |NULL|NULL|NULL|
|48382   |The Book of Mormon Movie, Volume 1: The Journey      |0000000000|NULL      |NULL  |NULL|NULL|NULL|
|325140  |Hum To Mohabbat Karega                               |0000000000|NULL      |NULL  |NULL|NULL|NULL|
|20653   |Roadside 

In [25]:
duplicate_crew = (
    crew_df.count()
    - crew_df.dropDuplicates().count()
)

print("Duplicate row numbers:", duplicate_crew)

Duplicate row numbers: 0


In [26]:
crew_df.show(5, truncate=False)

+--------+------+------------------------+----------+------+----+------------------------+-----------------+
|movie_id|title |credit_id               |department|gender|id  |job                     |name             |
+--------+------+------------------------+----------+------+----+------------------------+-----------------+
|19995   |Avatar|52fe48009251416c750aca23|Editing   |0     |1721|Editor                  |Stephen E. Rivkin|
|19995   |Avatar|539c47ecc3a36810e3001f87|Art       |2     |496 |Production Design       |Rick Carter      |
|19995   |Avatar|54491c89c3a3680fb4001cf7|Sound     |0     |900 |Sound Designer          |Christopher Boyes|
|19995   |Avatar|54491cb70e0a267480001bd0|Sound     |0     |900 |Supervising Sound Editor|Christopher Boyes|
|19995   |Avatar|539c4a4cc3a36810c9002101|Production|1     |1262|Casting                 |Mali Finn        |
+--------+------+------------------------+----------+------+----+------------------------+-----------------+
only showing top 5 

In [27]:
cast_df.show(5, truncate=False)

+--------+------+-------+-------------------+------------------------+------+-----+------------------+-----+
|movie_id|title |cast_id|character          |credit_id               |gender|id   |name              |order|
+--------+------+-------+-------------------+------------------------+------+-----+------------------+-----+
|19995   |Avatar|242    |Jake Sully         |5602a8a7c3a3685532001c9a|2     |65731|Sam Worthington   |0    |
|19995   |Avatar|3      |Neytiri            |52fe48009251416c750ac9cb|1     |8691 |Zoe Saldana       |1    |
|19995   |Avatar|25     |Dr. Grace Augustine|52fe48009251416c750aca39|1     |10205|Sigourney Weaver  |2    |
|19995   |Avatar|4      |Col. Quaritch      |52fe48009251416c750ac9cf|2     |32747|Stephen Lang      |3    |
|19995   |Avatar|5      |Trudy Chacon       |52fe48009251416c750ac9d3|1     |17647|Michelle Rodriguez|4    |
+--------+------+-------+-------------------+------------------------+------+-----+------------------+-----+
only showing top 5 

In [28]:
import boto3
from botocore.exceptions import ClientError

s3_client = boto3.client(
    "s3",
    endpoint_url=endpoint_url,
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key
)

bucket_name = "tmdb-silver"

try:
    s3_client.head_bucket(Bucket=bucket_name)
    print(f"{bucket_name} bucket exists.")

except ClientError:
    s3_client.create_bucket(Bucket=bucket_name)
    print(f"{bucket_name} bucket created.")

tmdb-silver bucket exists.


## Writing to RustFS

In [29]:
cast_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("s3a://tmdb-silver/cast")

In [30]:
crew_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("s3a://tmdb-silver/crew")

## Reading from RustFS

In [31]:
crew_delta_df = (
    spark.read
    .format("delta")
    .load("s3a://tmdb-silver/crew")
)

In [32]:
crew_delta_df.show(5, truncate=False)

26/07/05 15:58:03 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+--------+------+------------------------+----------+------+----+------------------------+-----------------+
|movie_id|title |credit_id               |department|gender|id  |job                     |name             |
+--------+------+------------------------+----------+------+----+------------------------+-----------------+
|19995   |Avatar|52fe48009251416c750aca23|Editing   |0     |1721|Editor                  |Stephen E. Rivkin|
|19995   |Avatar|539c47ecc3a36810e3001f87|Art       |2     |496 |Production Design       |Rick Carter      |
|19995   |Avatar|54491c89c3a3680fb4001cf7|Sound     |0     |900 |Sound Designer          |Christopher Boyes|
|19995   |Avatar|54491cb70e0a267480001bd0|Sound     |0     |900 |Supervising Sound Editor|Christopher Boyes|
|19995   |Avatar|539c4a4cc3a36810c9002101|Production|1     |1262|Casting                 |Mali Finn        |
+--------+------+------------------------+----------+------+----+------------------------+-----------------+
only showing top 5 

In [33]:
spark.stop()